In [17]:
import fastf1
import pandas as pd
import os
import socket
import logging
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline

logging.basicConfig(level=logging.INFO)

socket.setdefaulttimeout(20)

os.makedirs('cache', exist_ok=True)
fastf1.Cache.enable_cache('cache')

years = [2022, 2023, 2024, 2025]
all_laps = []

for year in years:
    try:
        session = fastf1.get_session(year, 'Belgium', 'R')
        session.load(telemetry=False, weather=True)
        laps = session.laps.copy()
        
        weather_df = laps.get_weather_data()
        laps['TrackTemp'] = weather_df['TrackTemp'].values
        laps['AirTemp'] = weather_df['AirTemp'].values
        
        laps = laps[(laps['IsAccurate'] == True) & (laps['TrackStatus'] == '1')]
        laps['LapTime_fc'] = laps['LapTime'].dt.total_seconds()
        
        laps['Year'] = year  
        all_laps.append(laps)
        
    except Exception as e:
        print(f"Failed to load {year}: {e}")

if all_laps:
    spa_laps = pd.concat(all_laps, ignore_index=True)
else:
    spa_laps = pd.DataFrame()


def find_dropoff_lap(stint_df, threshold_s=1.03, sustain_laps=2):
    stint_df = stint_df.sort_values('TyreLife').reset_index(drop=True)
    
    if len(stint_df) < 5:
        return None
    
    valid_stint = stint_df[stint_df['TyreLife'] > 2].copy()
    if valid_stint.empty:
        return None
        
    best_pace = valid_stint['LapTime_fc'].head(3).median()
    slow_mask = (valid_stint['LapTime_fc'] - best_pace) > threshold_s

    for i in range(len(slow_mask) - sustain_laps + 1):
        if slow_mask.iloc[i:i+sustain_laps].all():
            return valid_stint.iloc[i]['TyreLife']
            
    return None


records = []
if not spa_laps.empty:
    for (year, driver, stint_num), grp in spa_laps.groupby(['Year', 'Driver', 'Stint']):
        grp = grp.dropna(subset=['TyreLife', 'LapTime_fc'])
        if grp.empty:
            continue

        dropoff = find_dropoff_lap(grp)
        
        if dropoff is None or pd.isna(dropoff):
            continue  
            
        records.append({'Year': year, 'Driver': driver, 'Stint': stint_num, 'Compound': grp['Compound'].iloc[0], 'TrackTemp': grp['TrackTemp'].mean(), 'AirTemp': grp['AirTemp'].mean(), 'LapsUntilDropoff': dropoff})

model_df = pd.DataFrame(records)

if not model_df.empty:
    categorical = ['Driver', 'Compound']
    numeric = ['TrackTemp', 'AirTemp']

    model_df = model_df.dropna(subset=categorical + numeric + ['LapsUntilDropoff'])

    preprocessor = ColumnTransformer([('cat', OneHotEncoder(handle_unknown='ignore'), categorical)], remainder='passthrough')

    model = Pipeline([('prep', preprocessor), ('reg', LinearRegression())])

    X_train = model_df[categorical + numeric]
    y_train = model_df['LapsUntilDropoff']
    
    model.fit(X_train, y_train)

    def predict_laps(driver, compound, track_temp, air_temp, trained_model=model):
        input_df = pd.DataFrame([{'Driver': driver, 'Compound': compound, 'TrackTemp': track_temp, 'AirTemp': air_temp}])
        prediction = trained_model.predict(input_df)[0]
        return round(prediction, 1)

    predicted_laps = predict_laps('VER', 'hard', track_temp=30.0, air_temp=19.0)
    print(f"\nPredicted laps until drop-off: {predicted_laps}")
else:
    print("Not enough clean stint data found to build the model.")

core           INFO 	Loading data for Belgian Grand Prix - Race [v3.8.3]
INFO:fastf1.fastf1.core:Loading data for Belgian Grand Prix - Race [v3.8.3]
req            INFO 	Using cached data for session_info
INFO:fastf1.fastf1.req:Using cached data for session_info
req            INFO 	Using cached data for driver_info
INFO:fastf1.fastf1.req:Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
INFO:fastf1.fastf1.req:Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
INFO:fastf1.fastf1.req:Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
INFO:fastf1.fastf1.req:Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
INFO:fastf1.fastf1.req:Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
INFO:fastf1.fastf1.req:Using cached data for timing_app_data
core         


Predicted laps until drop-off: 9.6
